In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam

# Load Fashion MNIST Dataset

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Preprocessing Data

x_train = np.stack([x_train] * 3, axis=-1)
x_test = np.stack([x_test] * 3, axis=-1)

x_train = tf.image.resize(x_train, (128, 128))
x_test = tf.image.resize(x_test, (128, 128))

x_train = x_train.numpy()
x_test = x_test.numpy()

# Choose Pre-trained Model (VGG16)

base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(128, 128, 3)
)

# Freeze Base Layers

base_model.trainable = False

# Build Model

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

# Compile Model

model.compile(
    optimizer=Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train Model

history = model.fit(
    x_train,
    y_train,
    epochs=2,
    batch_size=64,
    validation_split=0.2
)

# Compute Loss and Evaluate Model

loss, accuracy = model.evaluate(x_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

# Plot Accuracy

plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Training and Validation Accuracy")
plt.legend(["Training", "Validation"])
plt.show()

# Plot Loss

plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Training and Validation Loss")
plt.legend(["Training", "Validation"])
plt.show()

# Visualize Predictions

class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

predictions = model.predict(x_test[:6])

plt.figure(figsize=(10, 6))

for i in range(6):
    plt.subplot(2, 3, i+1)

    plt.imshow(x_test[i][:, :, 0], cmap="gray")

    predicted = np.argmax(predictions[i])

    plt.title(
        f"Actual: {class_names[y_test[i]]}\n"
        f"Predicted: {class_names[predicted]}"
    )

    plt.axis("off")

plt.show()

# Fine Tune the Model

base_model.trainable = True

for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train Fine Tuned Model

fine_history = model.fit(
    x_train,
    y_train,
    epochs=2,
    batch_size=64,
    validation_split=0.2
)

# Evaluate Fine Tuned Model

loss, accuracy = model.evaluate(x_test, y_test)

print("Fine Tuned Test Loss:", loss)
print("Fine Tuned Test Accuracy:", accuracy)
